In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# dataset de brinquedo, duas imagens 4x4 com 1 canal só, uma classe 0 e outra classe 1
X_train = torch.tensor([[[[4, 5, 6, 7], [5, 6, 7, 8], [8, 9, 10, 11], [ 4, 5, 6, 7]]], [[[-4, 5, 6, -7], [ 5, -6, 7, 8], [-8, 9, -10, 11], [-4, -6, -7, -8]]]]).float().to(device)
X_train.div_(8)  # o "_" no final indica operação in-place -> divide os valores direto sem criar tensor novo (normalizando a escala)
y_train = torch.tensor([0, 1]).float().to(device)

In [ ]:
X_train.shape  # shape esperado pro Conv2d é (batch, canais, altura, largura) aqui (2,1,4,4)

torch.Size([2, 1, 4, 4])

In [59]:
y_train.shape

torch.Size([2])

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam

def get_model():
  model = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3), # convolução 1 canal de entrada, 1 filtro de saída, kernel 3x3
    nn.MaxPool2d(kernel_size=2), # reduz a dimensão espacial pela metade pegando o valor máximo de cada janela 2x2
    nn.ReLU(),
    nn.Flatten(), # achata tudo pra virar um vetor antes da camada densa
    nn.Linear(1,1),
    nn.Sigmoid() # sigmoid pq é classificação binária
  )

  loss_fn = nn.BCELoss()  # binary cross entropy, usa junto com Sigmoid
  optimizer = Adam(model.parameters(), lr=0.01)

  return model, loss_fn, optimizer

In [61]:
model, criterion, optimizer = get_model()

In [62]:
model

Sequential(
  (0): Conv2d(1, 1, kernel_size=(3, 3), stride=(1, 1))
  (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (2): ReLU()
  (3): Flatten(start_dim=1, end_dim=-1)
  (4): Linear(in_features=1, out_features=1, bias=True)
  (5): Sigmoid()
)

In [ ]:
import torch

def train_batch(x, y, model, optimizer, loss_fn):
  model.train()
  optimizer.zero_grad()
  prediction = model(x)
  batch_loss = loss_fn(prediction.squeeze(), y.squeeze()) # squeeze pra tirar dimensões extras de tamanho 1 e casar o shape da previsão com o do alvo
  batch_loss.backward()
  optimizer.step()
  return batch_loss.item()

In [ ]:
from torch.utils.data import TensorDataset, Dataset, DataLoader

trn_dl = DataLoader(TensorDataset(X_train, y_train))

In [ ]:
import torch

for epoch in range(2000):  # dataset bem pequeno então precisa de muitas epocas pra convergir
  for ix, batch in enumerate(trn_dl):
    x, y = batch
    x = x.to(device)
    y = y.to(device)
    batch_loss = train_batch(x, y, model, optimizer, criterion)

In [68]:
model(X_train[:1])  # testa a previsão pra primeira amostra

tensor([[0.0005]], grad_fn=<SigmoidBackward0>)